# Stream-CQSA in one notebook

A textbook attention kernel runs out of memory at a sequence length. The same
kernel, decomposed with a cyclic quorum set and recomposed, completes at that
length and returns the same numbers.

**Three cells.** Requirements, then every definition, then one call that runs
the whole thing and prints what happened. Nothing imports `stream_cqsa`; this is
the entire method in about 120 lines of PyTorch.

**You need** PyTorch and one CUDA GPU.

## 1. Requirements

In [1]:
# The only dependency is PyTorch with CUDA. On Colab or any ML image it is
# already there, so this normally just prints versions.
#
# If it is missing, install the build matching your CUDA and re-run:
#     %pip install torch --index-url https://download.pytorch.org/whl/cu128
#
# (Deliberately not auto-installed: pip will happily fetch a CPU-only wheel, or
# one built for the wrong CUDA, and this demo needs a working GPU.)
import torch
print(f"torch  : {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available(), "this demo needs a CUDA GPU"

torch  : 2.10.0+cu130
CUDA   : 13.0
GPU    : NVIDIA A100-SXM4-80GB


## How it works

Split the tokens into `C` chunks. Subproblem `i` gathers the chunks
`(i + offset) % C` for each offset in the interest set, so with `C=7` and
`(0, 1, 3)` subproblem 0 takes chunks 0, 1, 3; subproblem 1 takes 1, 2, 4; and so
on. Chunk `i` is subproblem `i`'s **owner**. Each subproblem attends only within
the tokens it gathered — `3/7` of the sequence here, so its score matrix is
`(3/7)²`, about a fifth of the original.

One rule makes the pieces fit: a subproblem skips any pair whose two ends lie in
the same chunk it does **not** own, because the subproblem that owns that chunk
will compute it. Every query–key pair is then computed exactly once.

The results cannot simply be added, because each subproblem's softmax is
normalized by a different denominator. So each returns a running maximum `m`, a
sum of exponentials `l`, and unnormalized weighted values `acc`. Merging rebases
both sides onto the larger maximum, which keeps every exponential in `(0, 1]` and
cannot overflow. Dividing `acc` by `l` at the end gives exactly the softmax the
whole sequence would have produced.

## 2. Everything, defined

In [2]:
# =============================================================================
# Stream-CQSA in full. Nothing below imports the `stream_cqsa` package -- this
# is the entire method in plain PyTorch, so you can read every step.
# =============================================================================
import gc, torch


# -----------------------------------------------------------------------------
# 1. Cutting the sequence up
# -----------------------------------------------------------------------------
def chunk_ids(n, c):
    """Which chunk each token belongs to. The first n % c chunks get one extra
    token, so this works for any n, not just multiples of c."""
    base, rem = divmod(n, c)
    out = []
    for i in range(c):
        out += [i] * (base + (1 if i < rem else 0))
    return torch.tensor(out, dtype=torch.long)


def subproblem(n, c, interest_set, i):
    """Subproblem i gathers chunks (i + offset) % c for each offset in the
    interest set. With c=7 and (0,1,3): subproblem 0 takes chunks 0,1,3;
    subproblem 1 takes 1,2,4; and so on. Chunk i is subproblem i's *owner*.

    Returns the global token indices it gathers (in sorted order), the chunk
    each of those tokens came from, and the owner chunk id."""
    cid = chunk_ids(n, c)
    keep = torch.zeros(n, dtype=torch.bool)
    for off in interest_set:
        keep |= (cid == (i + off) % c)
    tok = torch.nonzero(keep, as_tuple=True)[0]
    return tok, cid[tok], i


def pair_mask(local_chunk, owner, causal, tok):
    """The one rule that makes the pieces fit together.

    A subproblem skips any pair whose two ends sit in the same chunk that it
    does NOT own, because the subproblem that owns that chunk will compute it.
    Everything else it keeps. Across all c subproblems, every query-key pair is
    then computed exactly once -- no double counting, nothing dropped.

    Causality is applied on the original token positions, not local ones."""
    same = local_chunk[:, None] == local_chunk[None, :]
    both_away = (local_chunk != owner)[:, None] & (local_chunk != owner)[None, :]
    keep = ~(same & both_away)
    if causal:
        keep &= tok[:, None] >= tok[None, :]
    return keep


def coverage(n, c, interest_set, causal):
    """Count how many subproblems compute each pair. Every entry must be 1, or
    the decomposition is not a partition and the answer would be wrong."""
    cnt = torch.zeros(n, n, dtype=torch.int32)
    for i in range(c):
        tok, lc, owner = subproblem(n, c, interest_set, i)
        cnt[tok[:, None], tok[None, :]] += pair_mask(lc, owner, causal, tok).int()
    return cnt


# -----------------------------------------------------------------------------
# 2. The two kernels
# -----------------------------------------------------------------------------
def naive_attention(q, k, v, causal, log=lambda *_: None):
    """Textbook attention. Builds the whole N x N score matrix, so its memory
    grows with the square of the sequence length. That tensor is what fails."""
    B, H, N, D = q.shape
    log(f"building {N} x {N} scores ...")
    s = (q @ k.transpose(-2, -1)) * (D ** -0.5)
    if causal:
        s = s.masked_fill(torch.triu(torch.ones(N, N, dtype=torch.bool,
                                                device=q.device), 1), float("-inf"))
    log("softmax ...")
    return torch.softmax(s.float(), dim=-1).to(v.dtype) @ v


def cqsa_attention(q, k, v, causal, c, interest_set, log=lambda *_: None):
    """The same computation, one subproblem at a time.

    Each subproblem softmaxes over its own subset of keys, and those cannot
    simply be added: each is normalized by a different denominator. So a
    subproblem returns three running statistics instead of a finished output --
    the row maximum m, the sum of exponentials l, and the unnormalized weighted
    values acc. Merging rebases both sides onto the larger maximum, which keeps
    every exponential in (0, 1] and cannot overflow. Dividing acc by l at the
    very end gives exactly the softmax the whole sequence would have produced."""
    B, H, N, D = q.shape
    dev = q.device
    m   = torch.full((B, H, N), float("-inf"), device=dev, dtype=torch.float32)
    l   = torch.zeros((B, H, N),    device=dev, dtype=torch.float32)
    acc = torch.zeros((B, H, N, D), device=dev, dtype=torch.float32)

    for i in range(c):
        tok, lc, owner = subproblem(N, c, interest_set, i)
        n_i = tok.numel()
        log(f"subproblem {i+1}/{c}  gathering {n_i} of {N} tokens "
            f"(chunks {sorted((i + o) % c for o in interest_set)})  "
            f"scores {n_i*n_i*B*H*4/2**30:.2f} GiB")
        blocked = ~pair_mask(lc, owner, causal, tok).to(dev)
        tok = tok.to(dev)
        qi, ki, vi = q[:, :, tok], k[:, :, tok], v[:, :, tok]

        # Upcast the (small) operands rather than the (large) product, so an
        # fp16 score matrix and an fp32 copy are never both alive; then work in
        # place. Every extra copy here would cost another 1.5 GiB at N=16384.
        s = qi.float() @ ki.float().transpose(-2, -1)
        s *= (D ** -0.5)
        s.masked_fill_(blocked, float("-inf"))
        m_i = s.amax(-1)
        s.sub_(torch.nan_to_num(m_i, neginf=0.0)[..., None]).exp_()
        s.masked_fill_(blocked, 0.0)
        l_i, acc_i = s.sum(-1), s @ vi.float()
        del s, blocked, qi, ki, vi

        # merge onto the larger maximum: both scale factors are <= 1
        m_old = m[:, :, tok]
        m_new = torch.maximum(m_old, m_i)
        a = torch.exp(torch.nan_to_num(m_old - m_new, nan=0.0))
        b = torch.exp(torch.nan_to_num(m_i   - m_new, nan=0.0))
        l[:, :, tok]   = l[:, :, tok]   * a            + l_i   * b
        acc[:, :, tok] = acc[:, :, tok] * a[..., None] + acc_i * b[..., None]
        m[:, :, tok]   = m_new

    log("merging done, normalizing")
    return (acc / l.clamp(min=1e-20)[..., None]).to(q.dtype)


# -----------------------------------------------------------------------------
# 3. The whole demo, in one call
# -----------------------------------------------------------------------------
def run_demo(N=16384, C=7, INTEREST_SET=(0, 1, 3), B=1, H=8, D=64,
             causal=True, dtype=torch.float16, cap_gib=4.0):
    """Run both kernels at these settings and report everything."""
    assert torch.cuda.is_available(), "this demo needs a CUDA GPU"
    dev   = torch.device("cuda")
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    gib   = lambda x: x / 2**30

    def reset():
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    # Real cards vary, so cap this process instead. The demo then behaves the
    # same on a 24 GiB card and an 80 GiB one.
    torch.cuda.empty_cache()
    torch.cuda.set_per_process_memory_fraction(min(1.0, cap_gib / total))

    frac = len(INTEREST_SET) / C
    print("Stream-CQSA demo")
    print("=" * 78)
    print(f"device          {torch.cuda.get_device_name(0)} ({total:.1f} GiB), "
          f"capped to {cap_gib:.1f} GiB for this demo")
    print(f"problem         B={B} H={H} N={N} D={D} causal={causal} "
          f"{str(dtype).split('.')[-1]}")
    print(f"decomposition   C={C} interest_set={INTEREST_SET} -> {C} subproblems, "
          f"each gathering {len(INTEREST_SET)}/{C} of the sequence")

    # -- [1/4] is this decomposition valid at all? ----------------------------
    print(f"\n[1/4] checking the decomposition is a partition")
    n_s  = C * 9
    cnt  = coverage(n_s, C, INTEREST_SET, causal)
    want = (torch.tril(torch.ones(n_s, n_s, dtype=torch.int32)) if causal
            else torch.ones(n_s, n_s, dtype=torch.int32))
    if not torch.equal(cnt, want):
        print(f"      C={C} with {INTEREST_SET} is NOT a valid difference set:")
        print(f"        {(cnt < want).sum().item()} pairs would be dropped")
        print(f"        {(cnt > want).sum().item()} pairs would be counted twice")
        print(f"      results would be wrong, so stopping here.")
        return
    print(f"      every query-key pair is computed exactly once   OK")

    def make(n):
        reset(); torch.manual_seed(0)
        return tuple(torch.randn(B, H, n, D, device=dev, dtype=dtype) for _ in range(3))

    # -- [2/4] the kernel that fails ------------------------------------------
    print(f"\n[2/4] naive attention -- builds the full {N} x {N} score matrix")
    print(f"      that matrix alone is {gib(B*H*N*N*dtype.itemsize):.2f} GiB, and "
          f"softmax needs about twice that")
    q, k, v = make(N)
    naive_ok = True
    try:
        naive_attention(q, k, v, causal, log=lambda m: print(f"      {m}"))
        print(f"      completed, peak {gib(torch.cuda.max_memory_allocated()):.2f} GiB")
    except torch.cuda.OutOfMemoryError:
        naive_ok = False
        print(f"      OUT OF MEMORY   <- expected: over the {cap_gib:.1f} GiB cap")
    del q, k, v; reset()

    # -- [3/4] the same computation, decomposed -------------------------------
    print(f"\n[3/4] Stream-CQSA -- {C} subproblems, one at a time")
    q, k, v = make(N)
    try:
        cqsa_attention(q, k, v, causal, C, INTEREST_SET,
                       log=lambda m: print(f"      {m}"))
        peak = gib(torch.cuda.max_memory_allocated())
        print(f"      completed, peak {peak:.2f} GiB of the {cap_gib:.1f} GiB allowed")
        cqsa_ok = True
    except torch.cuda.OutOfMemoryError:
        peak, cqsa_ok = float("nan"), False
        print(f"      OUT OF MEMORY -- raise C (more, smaller subproblems) "
              f"or cap_gib")
    del q, k, v; reset()

    # -- [4/4] same numbers? --------------------------------------------------
    print(f"\n[4/4] do the two agree?")
    # Exactness is a property of the decomposition, not of the length, so a
    # small N is as good a check as a large one -- and unlike a large one, the
    # naive kernel is still available there to compare against.
    n_c = min(2048, N)
    if not naive_ok:
        print(f"      the naive kernel ran out of memory at N={N}, so there is no "
              f"reference to compare against there.")
    print(f"      comparing at N={n_c}, small enough that both fit "
          f"(exactness does not depend on N)")
    q, k, v = make(n_c)
    try:
        x = naive_attention(q, k, v, causal)
        y = cqsa_attention(q, k, v, causal, C, INTEREST_SET)
        rel = ((x.float() - y.float()).norm() / x.float().norm()).item()
        verdict = "same computation" if rel < 1e-2 else "MISMATCH"
        print(f"      relative difference {rel:.2e}   "
              f"(fp16 rounding is ~1e-03)   {verdict}")
    except torch.cuda.OutOfMemoryError:
        print(f"      even N={n_c} does not fit {cap_gib:.1f} GiB -- raise cap_gib")
    finally:
        del q, k, v; reset()

    print("\n" + "=" * 78)
    naive_txt = "ran out of memory" if not naive_ok else "completed"
    cqsa_txt  = f"completed in {peak:.2f} GiB" if cqsa_ok else "ran out of memory"
    print(f"summary   at N={N} under a {cap_gib:.1f} GiB budget: "
          f"naive {naive_txt}, Stream-CQSA {cqsa_txt}.")

## 3. Run it

Edit and re-run as often as you like. Valid `(C, INTEREST_SET)` pairs are
*perfect difference sets* — every pair of distinct chunks must meet in exactly
one subproblem:

| `C` | `INTEREST_SET` | each subproblem gathers |
|----:|:---------------|:------------------------|
| 3   | `(0, 1)`             | 2/3 of the sequence |
| 7   | `(0, 1, 3)`          | 3/7 |
| 13  | `(0, 1, 3, 9)`       | 4/13 |
| 21  | `(0, 1, 6, 8, 18)`   | 5/21 |

Larger `C` means more, smaller subproblems: less memory, more time. Trading one
for the other is the whole idea. Anything not in this table gets caught by the
partition check in step 1 — try `INTEREST_SET = (0, 1, 2)` to see it fire.

In [3]:
run_demo(
    N            = 16384,        # sequence length
    C            = 7,            # number of chunks
    INTEREST_SET = (0, 1, 3),    # which chunks each subproblem gathers
    B = 1, H = 8, D = 64,        # batch, heads, head dimension
    causal       = True,
    dtype        = torch.float16,
    cap_gib      = 4.0,          # pretend the card is this small
)

Stream-CQSA demo
device          NVIDIA A100-SXM4-80GB (79.3 GiB), capped to 4.0 GiB for this demo
problem         B=1 H=8 N=16384 D=64 causal=True float16
decomposition   C=7 interest_set=(0, 1, 3) -> 7 subproblems, each gathering 3/7 of the sequence

[1/4] checking the decomposition is a partition
      every query-key pair is computed exactly once   OK

[2/4] naive attention -- builds the full 16384 x 16384 score matrix
      that matrix alone is 4.00 GiB, and softmax needs about twice that


      building 16384 x 16384 scores ...
      OUT OF MEMORY   <- expected: over the 4.0 GiB cap

[3/4] Stream-CQSA -- 7 subproblems, one at a time
      subproblem 1/7  gathering 7023 of 16384 tokens (chunks [0, 1, 3])  scores 1.47 GiB


      subproblem 2/7  gathering 7022 of 16384 tokens (chunks [1, 2, 4])  scores 1.47 GiB
      subproblem 3/7  gathering 7022 of 16384 tokens (chunks [2, 3, 5])  scores 1.47 GiB
      subproblem 4/7  gathering 7021 of 16384 tokens (chunks [3, 4, 6])  scores 1.47 GiB
      subproblem 5/7  gathering 7021 of 16384 tokens (chunks [0, 4, 5])  scores 1.47 GiB
      subproblem 6/7  gathering 7021 of 16384 tokens (chunks [1, 5, 6])  scores 1.47 GiB


      subproblem 7/7  gathering 7022 of 16384 tokens (chunks [0, 2, 6])  scores 1.47 GiB
      merging done, normalizing
      completed, peak 1.66 GiB of the 4.0 GiB allowed

[4/4] do the two agree?
      the naive kernel ran out of memory at N=16384, so there is no reference to compare against there.
      comparing at N=2048, small enough that both fit (exactness does not depend on N)


      relative difference 4.90e-04   (fp16 rounding is ~1e-03)   same computation

summary   at N=16384 under a 4.0 GiB budget: naive ran out of memory, Stream-CQSA completed in 1.66 GiB.


## A word on the numbers here

The kernel in this notebook is deliberately naive. It still materializes a score
matrix, just a smaller one, so its memory falls only from `N²` to `(3/7·N)²` —
about a fifth. That is enough to make the point and small enough to read, but it
is a weak result, and raising `N` will hit the ceiling again after a couple of
doublings. That is a limit of this notebook, not of the method: when it happens,
raise `C`.

The shipped package does not materialize the score matrix at all. It uses a
FlashAttention-derived CUDA kernel, keeps Q, K and V in host memory, streams one
subproblem at a time to the device, and applies the decomposition recursively
when a single level is not enough. On one 80 GiB A100 that reaches
**N = 16,777,216**: the forward in **3,277 s** at 41.5 GiB peak, and a forward
and backward together in **15,847 s** at 70.4 GiB.

We stopped at 16M because one GPU and a fixed compute budget made that a natural
place to stop, not because the method stops there. Within the range we measured,
pushing further costs time rather than causing a failure. It is not unlimited,
though: some tensors stay proportional to `N` however finely the work is split.
At 16M the fp32 output alone is 32 GiB of that 41.5 GiB device peak, and Q, K and
V occupy 48 GiB of host memory. Those are what eventually bind.

Paper: [arXiv:2604.20819](https://doi.org/10.48550/arXiv.2604.20819) ·
Code: [github.com/yiming-b/Stream-CQSA](https://github.com/yiming-b/Stream-CQSA)